# Week 8: Comprehensive Evaluation Framework

This notebook implements the full Phase 4 evaluation with improvements:

1. **Scaled Benchmark** - 100+ examples across all categories/difficulties
2. **Improved Hallucination Detection** - Multi-signal approach
3. **Spike Detection Testing** - Validate uncertainty trigger metrics
4. **Comprehensive Experiments** - Full baseline comparison with statistical analysis

---

## Setup

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn sentence-transformers

In [ ]:
# Cell 2: Create directory structure
import os
import shutil

# Create module structure
os.makedirs('/content/orchestrator/entropy', exist_ok=True)
os.makedirs('/content/orchestrator/retrieval', exist_ok=True)
os.makedirs('/content/orchestrator/generation', exist_ok=True)
os.makedirs('/content/orchestrator/evaluation', exist_ok=True)

# Create root __init__.py
with open('/content/orchestrator/__init__.py', 'w') as f:
    f.write('"""Orchestrator package."""\n')

print("Directory structure created")

In [ ]:
# Cell 3: Upload module files
from google.colab import files

def upload_to_dir(target_dir):
    """Upload files and move to target directory."""
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.py'):
            dest = f'{target_dir}/{filename}'
            shutil.move(filename, dest)
            print(f"  OK {filename}")
    return uploaded

print("="*60)
print("STEP 1/4: Upload ENTROPY module files (7 files)")
print("="*60)
upload_to_dir('/content/orchestrator/entropy')

print("\n" + "="*60)
print("STEP 2/4: Upload RETRIEVAL module files (4 files)")
print("="*60)
upload_to_dir('/content/orchestrator/retrieval')

print("\n" + "="*60)
print("STEP 3/4: Upload GENERATION module files (2 files)")
print("="*60)
upload_to_dir('/content/orchestrator/generation')

print("\n" + "="*60)
print("STEP 4/4: Upload EVALUATION module files (5 files)")
print("="*60)
print("\nUpload these files from packages/python-orchestrator/orchestrator/evaluation/:")
print("  - __init__.py")
print("  - benchmark.py")
print("  - benchmark_generator.py  <-- NEW")
print("  - metrics.py")
print("  - runner.py")
upload_to_dir('/content/orchestrator/evaluation')

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)
!ls -la /content/orchestrator/evaluation/

In [ ]:
# Cell 4: Base imports
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any
from collections import defaultdict
import json
import time

sys.path.insert(0, '/content')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings('ignore')

print("Base imports complete")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Cell 5: Import evaluation modules
from orchestrator.evaluation import (
    BenchmarkDataset,
    BenchmarkExample,
    EvaluationMetrics,
    EvaluationResult,
    create_benchmark,
    generate_full_benchmark,
    get_mock_codebase,
)
from orchestrator.evaluation.benchmark import (
    Difficulty,
    Category,
)
from orchestrator.evaluation.runner import (
    BaselineRunner,
    ExperimentRunner,
    BaselineMethod,
)

print("All evaluation modules imported successfully!")
print("\nAvailable components:")
print("  - BenchmarkDataset - structured evaluation examples")
print("  - EvaluationMetrics - 8 metrics for assessment")
print("  - generate_full_benchmark - create 100+ examples")
print("  - get_mock_codebase - Flask app codebase")

---
## 1. Scaled Benchmark Dataset (100+ Examples)

In [ ]:
# Cell 6: Generate full benchmark
benchmark = generate_full_benchmark()

print("Full Benchmark Dataset")
print("="*60)
print(f"Name: {benchmark.name}")
print(f"Version: {benchmark.version}")
print(f"Total Examples: {len(benchmark)}")

stats = benchmark.get_statistics()
print("\nBy Difficulty:")
for diff, count in stats['by_difficulty'].items():
    print(f"  {diff}: {count}")

print("\nBy Category:")
for cat, count in stats['by_category'].items():
    print(f"  {cat}: {count}")

print(f"\nAverage ground truth files: {stats['avg_ground_truth_files']:.1f}")

In [ ]:
# Cell 7: Visualize benchmark distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By difficulty
ax1 = axes[0]
difficulties = list(stats['by_difficulty'].keys())
diff_counts = list(stats['by_difficulty'].values())
colors_diff = ['#2ecc71', '#f39c12', '#e74c3c']
bars1 = ax1.bar(difficulties, diff_counts, color=colors_diff[:len(difficulties)])
ax1.set_xlabel('Difficulty')
ax1.set_ylabel('Count')
ax1.set_title('Examples by Difficulty')
for bar, count in zip(bars1, diff_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', fontsize=12, fontweight='bold')

# By category
ax2 = axes[1]
categories = list(stats['by_category'].keys())
cat_counts = list(stats['by_category'].values())
colors_cat = plt.cm.Set3(np.linspace(0, 1, len(categories)))
bars2 = ax2.barh(categories, cat_counts, color=colors_cat)
ax2.set_xlabel('Count')
ax2.set_title('Examples by Category')
for bar, count in zip(bars2, cat_counts):
    ax2.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             str(count), ha='left', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('benchmark_distribution.png', dpi=150)
plt.show()
print("Saved: benchmark_distribution.png")

In [ ]:
# Cell 8: Sample examples from each category
print("Sample Examples from Each Category")
print("="*70)

for cat in Category:
    filtered = benchmark.filter(category=cat)
    if len(filtered) > 0:
        ex = filtered[0]
        print(f"\n[{cat.value.upper()}] {ex.id}")
        print(f"  Difficulty: {ex.difficulty.value}")
        print(f"  Query: {ex.query[:70]}...")
        print(f"  Files: {ex.ground_truth_files}")
        print(f"  Keywords: {ex.ground_truth_keywords[:5]}...")

---
## 2. Improved Hallucination Detection Testing

In [ ]:
# Cell 9: Initialize metrics
metrics = EvaluationMetrics(embedding_model='all-MiniLM-L6-v2')
print("EvaluationMetrics initialized with improved hallucination detection")

In [ ]:
# Cell 10: Test hallucination detection with clear examples

# Ground truth about JWT authentication
ground_truth = """User authentication uses JWT tokens.
The user logs in via /login endpoint.
Server creates JWT token with create_token() function.
Token uses HS256 algorithm with SECRET_KEY.
Token is verified using verify_token() on protected routes.
Token contains user_id, role, exp, and iat claims."""

keywords = ["JWT", "token", "login", "create_token", "verify_token", "HS256", "SECRET_KEY"]

# Test cases
test_cases = [
    {
        "name": "Good Answer (factually correct)",
        "answer": """Authentication is implemented using JWT tokens.
Users log in at the /login endpoint which calls create_token().
Tokens use HS256 algorithm and SECRET_KEY for signing.
Protected routes use verify_token() to authenticate requests.
The token payload includes user_id and role claims.""",
        "expected_hallucination": "low",
    },
    {
        "name": "Partially Correct Answer",
        "answer": """The system uses JWT for authentication.
Users log in via the /login endpoint.
Tokens are created with 256-bit encryption.
Sessions are stored in Redis for persistence.""",
        "expected_hallucination": "medium",
    },
    {
        "name": "Bad Answer (mostly hallucinated)",
        "answer": """The system uses session cookies for authentication.
Users are authenticated through OAuth2 with Google.
Passwords are stored in plain text for simplicity.
There is no token-based authentication system.
The API uses SOAP protocol for communication.""",
        "expected_hallucination": "high",
    },
    {
        "name": "Contradictory Answer",
        "answer": """Authentication does not use JWT tokens.
The system cannot verify tokens.
There is no login endpoint available.
Users never need to authenticate.""",
        "expected_hallucination": "high",
    },
]

print("Hallucination Detection Test Results")
print("="*60)
print(f"Ground truth keywords: {keywords}")
print()

hallucination_results = []
for tc in test_cases:
    rate = metrics.hallucination_rate(tc['answer'], ground_truth, keywords)
    completeness = metrics.answer_completeness(tc['answer'], ground_truth)
    correctness = metrics.answer_correctness(tc['answer'], ground_truth, keywords)

    hallucination_results.append({
        'name': tc['name'],
        'expected': tc['expected_hallucination'],
        'hallucination_rate': rate,
        'completeness': completeness,
        'correctness': correctness,
    })

    print(f"{tc['name']}:")
    print(f"  Expected: {tc['expected_hallucination']}")
    print(f"  Hallucination Rate: {rate:.3f}")
    print(f"  Completeness: {completeness:.3f}")
    print(f"  Correctness: {correctness:.3f}")
    print()

In [ ]:
# Cell 11: Visualize hallucination detection results
fig, ax = plt.subplots(figsize=(10, 6))

names = [r['name'] for r in hallucination_results]
x = np.arange(len(names))
width = 0.25

bars1 = ax.bar(x - width, [r['hallucination_rate'] for r in hallucination_results],
               width, label='Hallucination Rate', color='#e74c3c')
bars2 = ax.bar(x, [r['completeness'] for r in hallucination_results],
               width, label='Completeness', color='#3498db')
bars3 = ax.bar(x + width, [r['correctness'] for r in hallucination_results],
               width, label='Correctness', color='#2ecc71')

ax.set_ylabel('Score')
ax.set_title('Improved Hallucination Detection Performance')
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
ax.legend()
ax.set_ylim(0, 1.1)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold')

plt.tight_layout()
plt.savefig('hallucination_detection.png', dpi=150)
plt.show()
print("Saved: hallucination_detection.png")

---
## 3. Spike Detection Metric Testing

In [ ]:
# Cell 12: Test spike detection metrics
print("Spike Detection Metric Testing")
print("="*60)
print("\nSpike metrics measure how well uncertainty triggers align with")
print("positions where retrieval is actually needed.")
print()

# Test scenarios
spike_scenarios = [
    {
        "name": "Perfect Detection",
        "detected": [10, 25, 50, 75],
        "expected": [10, 25, 50, 75],
        "description": "All triggers exactly match expected positions",
    },
    {
        "name": "Good Detection (within tolerance)",
        "detected": [12, 24, 48, 77],
        "expected": [10, 25, 50, 75],
        "description": "Triggers within tolerance=3 of expected",
    },
    {
        "name": "Partial Detection (missed some)",
        "detected": [10, 50],
        "expected": [10, 25, 50, 75],
        "description": "Only detected 2 of 4 expected triggers",
    },
    {
        "name": "Over-triggering (false positives)",
        "detected": [10, 20, 25, 30, 50, 60, 75, 90],
        "expected": [10, 25, 50, 75],
        "description": "Detected all expected + 4 false positives",
    },
    {
        "name": "Poor Detection (misaligned)",
        "detected": [5, 35, 65, 95],
        "expected": [10, 25, 50, 75],
        "description": "Triggers far from expected positions",
    },
    {
        "name": "No Detection",
        "detected": [],
        "expected": [10, 25, 50, 75],
        "description": "System never triggered retrieval",
    },
]

spike_results = []
for scenario in spike_scenarios:
    precision = metrics.spike_precision(scenario['detected'], scenario['expected'])
    recall = metrics.spike_recall(scenario['detected'], scenario['expected'])
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    spike_results.append({
        'name': scenario['name'],
        'precision': precision,
        'recall': recall,
        'f1': f1,
    })

    print(f"{scenario['name']}:")
    print(f"  {scenario['description']}")
    print(f"  Detected: {scenario['detected']}")
    print(f"  Expected: {scenario['expected']}")
    print(f"  Precision: {precision:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}")
    print()

In [ ]:
# Cell 13: Visualize spike detection results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1 = axes[0]
names = [r['name'].replace(' ', '\n') for r in spike_results]
x = np.arange(len(names))
width = 0.25

ax1.bar(x - width, [r['precision'] for r in spike_results], width, label='Precision', color='#3498db')
ax1.bar(x, [r['recall'] for r in spike_results], width, label='Recall', color='#2ecc71')
ax1.bar(x + width, [r['f1'] for r in spike_results], width, label='F1', color='#9b59b6')

ax1.set_ylabel('Score')
ax1.set_title('Spike Detection Performance by Scenario')
ax1.set_xticks(x)
ax1.set_xticklabels(names, fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.1)

# Precision-Recall scatter
ax2 = axes[1]
for i, r in enumerate(spike_results):
    ax2.scatter(r['recall'], r['precision'], s=150, label=r['name'][:15])
    ax2.annotate(r['name'][:10], (r['recall']+0.02, r['precision']+0.02), fontsize=8)

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Spike Detection: Precision vs Recall')
ax2.set_xlim(-0.1, 1.1)
ax2.set_ylim(-0.1, 1.1)
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
ax2.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('spike_detection.png', dpi=150)
plt.show()
print("Saved: spike_detection.png")

---
## 4. Load Model and Run Comprehensive Experiments

In [ ]:
# Cell 14: Get mock codebase
documents = get_mock_codebase()

print(f"Mock Flask Application Codebase")
print("="*60)
print(f"Total files: {len(documents)}")
print(f"Total size: {sum(len(c) for c in documents.values()):,} characters")
print("\nFiles:")
for path in sorted(documents.keys()):
    print(f"  {path} ({len(documents[path]):,} chars)")

In [ ]:
# Cell 15: Load model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

print(f"Model loaded on {model.device}")

In [ ]:
# Cell 16: Import generation modules
from orchestrator.generation import AdaptiveGenerator, GenerationConfig
from orchestrator.retrieval.adaptive import SimpleRetriever

# Create retriever
retriever = SimpleRetriever(documents)

# Create adaptive generator
config = GenerationConfig(
    max_tokens=150,
    max_retrievals=3,
    uncertainty_threshold=2.5,
    cooldown_tokens=10,
)

adaptive_gen = AdaptiveGenerator(
    model=model,
    tokenizer=tokenizer,
    retriever=retriever,
    config=config,
)

print("AdaptiveGenerator configured")

In [ ]:
# Cell 17: Create baseline runner
baseline_runner = BaselineRunner(
    model=model,
    tokenizer=tokenizer,
    documents=documents,
)

print("BaselineRunner configured")

In [ ]:
# Cell 18: Run comprehensive experiments
# Select a subset of examples for faster evaluation
# (Use full benchmark for final experiments)

# Get balanced sample: 2 from each category
sample_examples = []
for cat in Category:
    filtered = benchmark.filter(category=cat)
    for i, ex in enumerate(filtered):
        if i < 2:  # 2 per category
            sample_examples.append(ex)

print(f"Running experiments on {len(sample_examples)} examples")
print(f"Categories covered: {len(set(ex.category for ex in sample_examples))}")

# Methods to evaluate
methods = [
    BaselineMethod.NO_CONTEXT,
    BaselineMethod.BM25,
    BaselineMethod.EMBEDDING,
    BaselineMethod.FULL_CONTEXT,
]

# Results storage
all_results = {m.value: [] for m in methods}
all_results['cce_adaptive'] = []

# Compute baseline tokens
baseline_tokens = sum(len(tokenizer.encode(c)) for c in documents.values())
print(f"Baseline tokens (full context): {baseline_tokens:,}")
print()

# Run experiments
total = len(sample_examples) * (len(methods) + 1)  # +1 for CCE
current = 0

for i, example in enumerate(sample_examples):
    print(f"\nExample {i+1}/{len(sample_examples)}: {example.id}")
    print(f"  Category: {example.category.value}")
    print(f"  Query: {example.query[:50]}...")

    # Run baselines
    for method in methods:
        current += 1
        print(f"  Running {method.value}... ({current}/{total})")

        result = baseline_runner.run(example, method, max_tokens=150)

        eval_result = metrics.evaluate(
            example_id=example.id,
            generated_answer=result['answer'],
            ground_truth_answer=example.ground_truth_answer,
            retrieved_files=result['retrieved_files'],
            ground_truth_files=example.ground_truth_files,
            ground_truth_keywords=example.ground_truth_keywords,
            tokens_used=result['tokens_used'],
            baseline_tokens=baseline_tokens,
            generation_time=result['generation_time'],
            num_retrievals=result['num_retrievals'],
            method=method.value,
        )
        eval_result.metadata['category'] = example.category.value
        eval_result.metadata['difficulty'] = example.difficulty.value

        all_results[method.value].append(eval_result)

    # Run CCE adaptive
    current += 1
    print(f"  Running cce_adaptive... ({current}/{total})")

    try:
        gen_result = adaptive_gen.generate(
            query=example.query,
            initial_context="This is a Flask web application.",
        )

        cce_eval = metrics.evaluate(
            example_id=example.id,
            generated_answer=gen_result.response,
            ground_truth_answer=example.ground_truth_answer,
            retrieved_files=gen_result.context_sources,
            ground_truth_files=example.ground_truth_files,
            ground_truth_keywords=example.ground_truth_keywords,
            tokens_used=gen_result.total_context_tokens,
            baseline_tokens=baseline_tokens,
            generation_time=gen_result.generation_time,
            num_retrievals=gen_result.total_retrievals,
            method='cce_adaptive',
        )
        cce_eval.metadata['category'] = example.category.value
        cce_eval.metadata['difficulty'] = example.difficulty.value
        all_results['cce_adaptive'].append(cce_eval)
    except Exception as e:
        print(f"  CCE error: {e}")

print("\n" + "="*60)
print("Experiments Complete!")

In [ ]:
# Cell 19: Aggregate results
print("Aggregate Results")
print("="*70)

aggregates = {}
for method, results in all_results.items():
    if results:
        aggregates[method] = metrics.aggregate_results(results)

# Create comparison table
comparison_data = []
for method, agg in aggregates.items():
    row = {
        'Method': method,
        'Correctness': agg['metrics']['answer_correctness']['mean'],
        'Completeness': agg['metrics']['answer_completeness']['mean'],
        'Hallucination': agg['metrics']['hallucination_rate']['mean'],
        'Context P': agg['metrics']['context_precision']['mean'],
        'Context R': agg['metrics']['context_recall']['mean'],
        'Efficiency': agg['metrics']['token_efficiency']['mean'],
        'Composite': agg['metrics']['composite_score']['mean'],
    }
    comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.sort_values('Composite', ascending=False)

print(df_comparison.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# Cell 20: Visualize comprehensive results
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

methods_order = df_comparison['Method'].tolist()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# 1. Composite scores
ax1 = axes[0, 0]
bars = ax1.bar(methods_order, df_comparison['Composite'], color=colors[:len(methods_order)])
ax1.set_ylabel('Composite Score')
ax1.set_title('Overall Performance (Composite Score)')
ax1.set_ylim(0, 1)
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{bar.get_height():.3f}', ha='center', fontsize=10)

# 2. Metric comparison radar-style
ax2 = axes[0, 1]
metric_cols = ['Correctness', 'Completeness', 'Context P', 'Context R', 'Efficiency']
for i, method in enumerate(methods_order):
    row = df_comparison[df_comparison['Method'] == method].iloc[0]
    values = [row[m] for m in metric_cols]
    ax2.plot(metric_cols, values, 'o-', label=method, color=colors[i % len(colors)])
ax2.set_ylabel('Score')
ax2.set_title('Metrics Comparison')
ax2.legend(loc='lower right')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 3. Hallucination rate (lower is better)
ax3 = axes[1, 0]
halluc_values = df_comparison['Hallucination'].tolist()
bars3 = ax3.bar(methods_order, halluc_values, color=['#e74c3c' if v > 0.3 else '#2ecc71' for v in halluc_values])
ax3.set_ylabel('Hallucination Rate')
ax3.set_title('Hallucination Rate (Lower is Better)')
ax3.set_ylim(0, 1)
ax3.axhline(y=0.3, color='orange', linestyle='--', alpha=0.7, label='Threshold')
for bar in bars3:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{bar.get_height():.3f}', ha='center', fontsize=10)

# 4. Efficiency vs Quality tradeoff
ax4 = axes[1, 1]
for i, method in enumerate(methods_order):
    row = df_comparison[df_comparison['Method'] == method].iloc[0]
    ax4.scatter(row['Efficiency'], row['Correctness'], s=200,
                color=colors[i % len(colors)], label=method)
ax4.set_xlabel('Token Efficiency')
ax4.set_ylabel('Answer Correctness')
ax4.set_title('Quality vs Efficiency Tradeoff')
ax4.legend(loc='lower left')
ax4.set_xlim(0, 1.1)
ax4.set_ylim(0, 1.1)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comprehensive_results.png', dpi=150)
plt.show()
print("Saved: comprehensive_results.png")

In [ ]:
# Cell 21: Results by category
print("Results by Category")
print("="*70)

# Organize results by category
results_by_category = defaultdict(lambda: defaultdict(list))

for method, results in all_results.items():
    for r in results:
        cat = r.metadata.get('category', 'unknown')
        results_by_category[cat][method].append(r.get_composite_score())

# Create category comparison table
cat_data = []
for cat in Category:
    row = {'Category': cat.value}
    for method in ['no_context', 'bm25', 'embedding', 'full_context', 'cce_adaptive']:
        scores = results_by_category[cat.value].get(method, [])
        row[method] = np.mean(scores) if scores else 0.0
    cat_data.append(row)

df_category = pd.DataFrame(cat_data)
print(df_category.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# Cell 22: Category heatmap
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data for heatmap
method_cols = [c for c in df_category.columns if c != 'Category']
heatmap_data = df_category[method_cols].values

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    xticklabels=method_cols,
    yticklabels=df_category['Category'],
    ax=ax,
    vmin=0,
    vmax=1,
)

ax.set_title('Composite Score by Category and Method')
plt.tight_layout()
plt.savefig('category_heatmap.png', dpi=150)
plt.show()
print("Saved: category_heatmap.png")

In [ ]:
# Cell 23: Statistical significance testing
from scipy import stats

print("Statistical Significance Testing")
print("="*70)
print("\nComparing CCE Adaptive against baselines (paired t-test)")
print()

cce_scores = [r.get_composite_score() for r in all_results.get('cce_adaptive', [])]

if cce_scores:
    for method in ['no_context', 'bm25', 'embedding', 'full_context']:
        baseline_scores = [r.get_composite_score() for r in all_results.get(method, [])]

        if len(baseline_scores) == len(cce_scores):
            t_stat, p_value = stats.ttest_rel(cce_scores, baseline_scores)
            mean_diff = np.mean(cce_scores) - np.mean(baseline_scores)

            print(f"CCE vs {method}:")
            print(f"  Mean difference: {mean_diff:+.3f}")
            print(f"  t-statistic: {t_stat:.3f}")
            print(f"  p-value: {p_value:.4f}")
            print(f"  Significant (p<0.05): {'Yes' if p_value < 0.05 else 'No'}")
            print()
else:
    print("No CCE results available for comparison")

---
## 5. Summary and Export

In [ ]:
# Cell 24: Summary
print("="*70)
print("WEEK 8 COMPREHENSIVE EVALUATION COMPLETE")
print("="*70)

print('''
IMPROVEMENTS IMPLEMENTED:

1. SCALED BENCHMARK (100+ examples)
   - 6 categories: architecture, api_usage, implementation,
     debugging, configuration, data_flow
   - 3 difficulties: easy, medium, hard
   - Mock Flask codebase with 12+ files

2. IMPROVED HALLUCINATION DETECTION
   - Multi-signal approach:
     * Keyword overlap (30%)
     * Entity/fact extraction (30%)
     * Semantic similarity (40%)
   - Contradiction detection with penalty
   - Stricter threshold (0.3 support score)

3. SPIKE DETECTION METRICS TESTED
   - Precision: accuracy of triggers
   - Recall: coverage of expected triggers
   - Tolerance-based matching (default: 3 tokens)

4. COMPREHENSIVE EXPERIMENTS
   - 5 methods compared: no_context, bm25, embedding,
     full_context, cce_adaptive
   - Results by category and difficulty
   - Statistical significance testing

FIGURES GENERATED:
  - benchmark_distribution.png
  - hallucination_detection.png
  - spike_detection.png
  - comprehensive_results.png
  - category_heatmap.png
''')

# Print best method
if not df_comparison.empty:
    best = df_comparison.iloc[0]
    print(f"BEST METHOD: {best['Method']}")
    print(f"  Composite Score: {best['Composite']:.3f}")
    print(f"  Correctness: {best['Correctness']:.3f}")
    print(f"  Efficiency: {best['Efficiency']:.3f}")

In [ ]:
# Cell 25: Export results
results_export = {
    'benchmark_stats': stats,
    'comparison_table': df_comparison.to_dict('records'),
    'category_results': df_category.to_dict('records'),
    'hallucination_tests': hallucination_results,
    'spike_detection_tests': spike_results,
}

with open('week8_results.json', 'w') as f:
    json.dump(results_export, f, indent=2, default=str)

print("Results exported to week8_results.json")

# Download files
try:
    from google.colab import files
    for fname in ['benchmark_distribution.png', 'hallucination_detection.png',
                  'spike_detection.png', 'comprehensive_results.png',
                  'category_heatmap.png', 'week8_results.json']:
        if os.path.exists(fname):
            files.download(fname)
except:
    print("Download files manually or run locally")